[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 06](README.md)

# CUDA: modelo SIMT, grid y memoria

**Tema:** 06 · **Sesiones:** 25, 26 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo mapear un dominio a grid/bloque/hilo y medir el costo completo con errores comprobados?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** CUDA hace explícita la jerarquía de ejecución. Un mapeo correcto cubre el dominio, controla bordes y separa errores de lanzamiento de errores asíncronos.

**Prerrequisitos.**

- C++20, memoria y descomposición por datos.
- Modelo host–dispositivo y medición extremo a extremo.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Explicar host, device, kernel, warp, bloque y grid.
- Calcular cobertura con guardas de borde.
- Separar transferencia, kernel y tiempo extremo a extremo.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Los hilos de un warp siguen un modelo SIMT; divergencia serializa caminos dentro del warp.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Los bloques deben ser independientes salvo coordinación mediante lanzamientos separados o mecanismos específicos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Los errores de lanzamiento y los errores asíncronos se comprueban en puntos distintos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- SIMT — ejecución de hilos agrupados sobre una instrucción
- warp — grupo de hilos planificado conjuntamente
- coalescencia — agrupación eficiente de accesos contiguos
- tile — bloque de datos reutilizado localmente


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Cuda Grid Tiling

![Jerarquía grid–bloque–hilo y tile compartido](../../images/cuda-grid-tiling.svg)

**Cómo leerlo.** Primero ubica un hilo dentro de su bloque y grid; después observa que la cooperación y sincronización ocurren dentro del bloque que reutiliza el tile.

### Offload Host Device

![Flujo de datos entre host y dispositivo](../../images/offload-host-device.svg)

**Cómo leerlo.** Separa preparación, H2D, kernel, D2H y validación. Esa separación evita llamar tiempo total a una medición que solo cubre el kernel.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "06"
NOTEBOOK = "06_cuda/01_modelo_cuda.ipynb"
assert (ROOT / "curso" / "notebooks" / "06_cuda" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Cobertura del grid

**Situación.** Se calcula el número de bloques y se prueban tamaños no múltiplos.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def launch_shape(n, block):
    grid = (n + block - 1) // block
    launched = grid * block
    return {"n": n, "block": block, "grid": grid, "launched": launched, "guarded": launched-n}
for n in (1, 255, 256, 257, 1000, 1_000_003):
    row = launch_shape(n, 256)
    assert row["launched"] >= n and row["guarded"] < 256
    print(row)


### Explicación del resultado

Cada kernel usa `if (i<n)` cuando la geometría lanza hilos adicionales.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Descomposición temporal

**Situación.** Se evita atribuir al kernel los costos de preparación y copia.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
runs = {"alloc": 0.18, "h2d": 1.45, "kernel": 0.62, "d2h": 1.10, "sync": 0.08}
total = sum(runs.values())
for phase, elapsed in runs.items(): print(f"{phase:8} {elapsed:5.2f} ms {100*elapsed/total:5.1f}%")
print("total", round(total, 3), "ms")
assert total > runs["kernel"]


### Lectura razonada

Eventos CUDA miden trabajo en streams; un reloj de host delimita el tiempo extremo a extremo con sincronización explícita.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Por qué una geometría que lanza más hilos que elementos necesita una guarda aunque el cálculo de bloques sea correcto?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Construir vector add y comparar contra CPU.
2. Probar tamaños 0/1, no múltiplos y grandes.
3. Ejecutar Compute Sanitizer y conservar dispositivo/toolkit.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Omitir la guarda de borde.
- Leer resultados antes de sincronizar.
- Comprobar solo `cudaGetLastError` y no el trabajo asíncrono.


## Criterios de aceptación

- Máximo error dentro de tolerancia.
- Todos los estados CUDA comprobados.
- Kernel y total reportados por separado.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo mapear un dominio a grid/bloque/hilo y medir el costo completo con errores comprobados?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Guía CUDA](README.md)
- [Ejemplos CUDA](../../ejemplos/06_cuda/README.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 06](README.md)
